In [147]:

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import json
import joblib
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

# ============================================================
# 2. LOAD THE CLEANED DATASETS
# Run this notebook from the ml folder, as before.
# ============================================================

DATA_PATH = Path("../data/clean")

customers = pd.read_csv(DATA_PATH / "customers_clean.csv")
loans = pd.read_csv(DATA_PATH / "loans_clean.csv")

print("Customers:", customers.shape)
print("Loans:", loans.shape)

# ============================================================
# 3. MERGE LOANS WITH CUSTOMER INFORMATION
# ============================================================

customer_features = customers[
    [
        "customer_id",
        "age",
        "annual_income",
        "occupation",
        "marital_status",
        "customer_segment"
    ]
]

ml_data = loans.merge(
    customer_features,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

print("Combined dataset:", ml_data.shape)

# ============================================================
# 4. SELECT FEATURES AND TARGET
# ============================================================

numeric_features = [
    "loan_amount",
    "interest_rate",
    "emi",
    "credit_score",
    "age",
    "annual_income"
]

categorical_features = [
    "loan_type",
    "occupation",
    "marital_status",
    "customer_segment"
]

feature_columns = numeric_features + categorical_features

X = ml_data[feature_columns].copy()
y = ml_data["default_flag"].map({"No": 0, "Yes": 1})

if y.isna().any():
    raise ValueError("Unexpected values found in default_flag.")

if X.isna().any().any():
    raise ValueError("Missing values found in model features.")

print("Features:", len(feature_columns))
print("Default rate:", round(y.mean() * 100, 2), "%")

# ============================================================
# 5. SPLIT DATA INTO TRAINING AND TEST SETS
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

# ============================================================
# 6. CREATE PREPROCESSING PIPELINE
# ============================================================

def create_preprocessor():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ])

# ============================================================
# 7. DEFINE BOTH CLASSIFICATION MODELS
# ============================================================

model_configs = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Random Forest Classifier": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

trained_models = {}
comparison_results = {}

# ============================================================
# 8. TRAIN AND EVALUATE BOTH MODELS
# ============================================================

for name, classifier in model_configs.items():

    print("\n" + "=" * 55)
    print(name)
    print("=" * 55)

    pipeline = Pipeline([
        ("preprocessor", create_preprocessor()),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train, y_train)

    probabilities = pipeline.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.50).astype(int)

    trained_models[name] = pipeline

    comparison_results[name] = {
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_test, predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "Average Precision": average_precision_score(
            y_test, probabilities
        ),
        "Brier Score": brier_score_loss(
            y_test, probabilities
        )
    }

    print("\nClassification report:")
    print(classification_report(
        y_test,
        predictions,
        labels=[0, 1],
        target_names=["No default", "Default"],
        zero_division=0
    ))

    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions))

# ============================================================
# 9. COMPARE THE MODELS
# ============================================================

comparison = pd.DataFrame(comparison_results).T.round(4)

print("\nMODEL COMPARISON AT THRESHOLD 0.50")
print(comparison)

# Save comparison results
ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

comparison.to_csv(
    ARTIFACTS_DIR / "model_comparison.csv"
)

# ============================================================
# 10. SAVE BOTH TRAINED MODELS
# ============================================================

joblib.dump(
    trained_models["Logistic Regression"],
    ARTIFACTS_DIR / "logistic_regression_model.joblib"
)

joblib.dump(
    trained_models["Random Forest Classifier"],
    ARTIFACTS_DIR / "random_forest_model.joblib"
)

print("\nBoth trained models saved.")

# ============================================================
# 11. PREPARE THE MODEL FOR THE EXISTING API
# ============================================================

# Provisional API baseline:
# Logistic Regression with the previously explored threshold of 0.10.
# This threshold needs independent validation before production use.

selected_model = trained_models["Logistic Regression"]
selected_threshold = 0.10

selected_probabilities = selected_model.predict_proba(X_test)[:, 1]
selected_predictions = (
    selected_probabilities >= selected_threshold
).astype(int)

print("\nLOGISTIC REGRESSION AT THRESHOLD 0.10")
print(classification_report(
    y_test,
    selected_predictions,
    labels=[0, 1],
    target_names=["No default", "Default"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test, selected_predictions))

# Save the API-compatible model
joblib.dump(
    selected_model,
    ARTIFACTS_DIR / "loan_default_model.joblib"
)

# Save metadata used by api/app.py
metadata = {
    "model_name": "Logistic Regression",
    "target": "default_flag",
    "threshold": selected_threshold,
    "features": feature_columns,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "positive_class": 1,
    "threshold_status": "provisional; requires validation"
}

with open(
    ARTIFACTS_DIR / "model_metadata.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(metadata, file, indent=4)

print("\nSaved files:")
for path in sorted(ARTIFACTS_DIR.iterdir()):
    if path.is_file():
        print(path)


Customers: (75000, 13)
Loans: (30000, 10)
Combined dataset: (30000, 15)
Features: 10
Default rate: 5.39 %
Training rows: 24000
Testing rows: 6000

Logistic Regression

Classification report:
              precision    recall  f1-score   support

  No default       0.95      1.00      0.97      5677
     Default       0.09      0.00      0.01       323

    accuracy                           0.94      6000
   macro avg       0.52      0.50      0.49      6000
weighted avg       0.90      0.94      0.92      6000

Confusion matrix:
[[5667   10]
 [ 322    1]]

Random Forest Classifier

Classification report:
              precision    recall  f1-score   support

  No default       0.95      0.98      0.97      5677
     Default       0.21      0.07      0.11       323

    accuracy                           0.94      6000
   macro avg       0.58      0.53      0.54      6000
weighted avg       0.91      0.94      0.92      6000

Confusion matrix:
[[5589   88]
 [ 299   24]]

MODEL COMPARIS